# **Regression: Lasso Regression (L1 Regularization)**

## **Justification of Preprocessing Strategy**

### **The Absolute Necessity of Scaling for L1 Penalty**
**Lasso Regression** introduces an L1 regularization penalty to the Ordinary Least Squares (OLS) loss function, adding the sum of the absolute values of the coefficients to the optimization target.

Because the penalty directly restricts the absolute magnitude of the $\beta$ weights, features with larger underlying numerical ranges will naturally have smaller coefficients to compensate, making them the target of unfair penalization. To prevent the algorithm from erroneously suppressing clinically significant variables, the feature space must be uniform. We will evaluate both **Standardization (StandardScaler)** and **Normalization (MinMaxScaler)** across all optimization levels to discover which framework yields the most accurate predictions for the continuous `diabetes_risk_score`.

### **Automated Feature Selection and Sparsity**
The unique mathematical trait of L1 regularization is its ability to force less important feature coefficients to become **exactly zero**. In a dataset containing 100,000 samples and numerous dummy-encoded variables, Lasso acts as an embedded feature selection tool, filtering out noise and tackling multicollinearity. To avoid data leakage, we drop categorical targets (`diagnosed_diabetes`, `diabetes_stage`), and we **omit stratification** during the split since we are modeling a continuous numerical distribution.


## **Experiment Design**

We have designed a complete tournament consisting of **6 distinct runs** to evaluate both scalers across 3 optimization levels, using **MAE, RMSE, and $R^2$** as performance indicators:

* **Run 1 & 2: Lasso Baseline** — Testing the Lasso model with strict Scikit-Learn default parameters under **Standardization** vs. **Normalization**.
* **Run 3 & 4: GridSearchCV Tuning** — Performing an exhaustive search over a fixed grid of the `alpha` parameter under **Standardization** vs. **Normalization**.
* **Run 5 & 6: Optuna Optimization** — Utilizing Bayesian optimization to fine-tune both `alpha` and `max_iter` continuously under **Standardization** vs. **Normalization**.

In all optimization runs (GridSearchCV and Optuna), trials are evaluated using **3-Fold Cross-Validation** to guarantee model generalizability.


In [2]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_Lasso")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Drop classification targets to avoid data leakage
X = df_final.drop(["diabetes_risk_score", "diagnosed_diabetes", "diabetes_stage"], axis=1, errors='ignore')
y = df_final['diabetes_risk_score']

# Split data (80/20) - Continuous target means NO stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_regression_metrics(y_true, y_pred, duration):
    """Utility function to log regression metrics to MLflow"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_metric("fit_time", duration)

# Define the scaling strategies to compare across the entire tournament
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

# ---------------------------------------------------------
# RUN 1 & 2: LASSO BASELINE (Strict Defaults)
# ---------------------------------------------------------
for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Lasso_Baseline_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        model = Lasso(random_state=42)
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        mlflow.log_param("optimization", "none_default")
        mlflow.log_param("scaler", s_name)
        mlflow.log_params(model.get_params())
        
        log_regression_metrics(y_test, model.predict(X_test_scaled), duration)

# ---------------------------------------------------------
# RUN 3 & 4: GRIDSEARCHCV TUNING
# ---------------------------------------------------------
param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]}

for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Lasso_GridSearch_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        grid = GridSearchCV(
            Lasso(random_state=42, max_iter=2000), 
            param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        start_time = time.time()
        grid.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        best_model = grid.best_estimator_
        zero_coefs = np.sum(best_model.coef_ == 0)
        
        mlflow.log_param("optimization", "GridSearchCV")
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("eliminated_features", f"{zero_coefs}/{len(best_model.coef_)}")
        mlflow.log_params(grid.best_params_)
        
        log_regression_metrics(y_test, best_model.predict(X_test_scaled), duration)

# ---------------------------------------------------------
#  RUN 5 & 6: OPTUNA BAYESIAN OPTIMIZATION
# ---------------------------------------------------------
for s_name, scaler_obj in scalers.items():
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
    
    def objective(trial):
        alpha_val = trial.suggest_float("alpha", 1e-5, 5.0, log=True)
        max_iter_val = trial.suggest_int("max_iter", 1000, 4000)
        model = Lasso(alpha=alpha_val, max_iter=max_iter_val, random_state=42)
        # Target: Minimize MAE (negate negative MAE back to positive)
        scores = -cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1).mean()
        return scores.mean()

    with mlflow.start_run(run_name=f"Lasso_Optuna_{s_name}"):
        study = optuna.create_study(direction="minimize")
        start_time = time.time()
        study.optimize(objective, n_trials=15)
        duration = time.time() - start_time
        
        best_lasso_opt = Lasso(**study.best_params, random_state=42)
        best_lasso_opt.fit(X_train_scaled, y_train)
        zero_coefs_opt = np.sum(best_lasso_opt.coef_ == 0)
        
        mlflow.log_param("optimization", "optuna")
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("eliminated_features", f"{zero_coefs_opt}/{len(best_lasso_opt.coef_)}")
        mlflow.log_params(study.best_params)
        
        log_regression_metrics(y_test, best_lasso_opt.predict(X_test_scaled), duration)

2026/05/20 09:39:15 INFO mlflow.tracking.fluent: Experiment with name 'Regression_Lasso' does not exist. Creating a new experiment.
[I 2026-05-20 09:39:48,749] A new study created in memory with name: no-name-51cd586f-da81-4113-a380-3c69e45f87bb
[I 2026-05-20 09:39:49,994] Trial 0 finished with value: 0.39711107607149043 and parameters: {'alpha': 0.0003888169453778615, 'max_iter': 3561}. Best is trial 0 with value: 0.39711107607149043.
[I 2026-05-20 09:39:50,507] Trial 1 finished with value: 0.5971331142278209 and parameters: {'alpha': 0.16825951639178163, 'max_iter': 2431}. Best is trial 0 with value: 0.39711107607149043.
[I 2026-05-20 09:39:57,083] Trial 2 finished with value: 0.39704224153894874 and parameters: {'alpha': 0.00011100467121479616, 'max_iter': 3523}. Best is trial 2 with value: 0.39704224153894874.
[I 2026-05-20 09:39:58,062] Trial 3 finished with value: 0.39741370555039773 and parameters: {'alpha': 0.0010332564166458166, 'max_iter': 2756}. Best is trial 2 with value: 0

## **Winner Run Selection (Priority Elimination Framework)**

### **Selection Criteria (in priority order)**
1. **Priority 1 (60% weight): Lowest MAE** — Clinical proximity; minimizes average day-to-day prediction error.
2. **Priority 2 (30% weight): RMSE proportional to MAE** — Rejects runs where RMSE spikes relative to MAE, indicating catastrophic errors.
3. **Priority 3 (10% weight): Acceptable R²** — Confirms statistical fit quality.
4. **Tiebreaker: Lowest Fit Time** — Applied only if a technical tie exists in MAE, RMSE, and R².

### **All Runs: Summary Table with Metrics**

| Run | Optimization | Scaler | Alpha | Eliminated Features | MAE | RMSE | R² Score | Fit Time (s) |
|---|---|---|---:|---:|---:|---:|---:|---:|
| **Lasso_GridSearch_Standardization** | **GridSearchCV** | **Standardization** | **0.0001** | **6/53** | **0.4039** | **0.7113** | **0.9939** | **25.50** |
| Lasso_Optuna_Standardization | Optuna | Standardization | 3.70e-05 | 3/53 | 0.4039 | 0.7113 | 0.9939 | 74.77 |
| Lasso_Optuna_Normalization | Optuna | Normalization | 1.06e-05 | 5/53 | 0.4040 | 0.7113 | 0.9939 | 30.73 |
| Lasso_GridSearch_Normalization | GridSearchCV | Normalization | 0.0001 | 12/53 | 0.4044 | 0.7113 | 0.9939 | 5.81 |
| Lasso_Baseline_Standardization | None (Default) | Standardization | 1.0 | 0/53 | 2.0209 | 2.5227 | 0.9229 | 0.23 |
| Lasso_Baseline_Normalization | None (Default) | Normalization | 1.0 | 0/53 | 5.0730 | 6.3322 | 0.5141 | 0.19 |

### **Step-by-Step Elimination Process**

**Step 1: Filter by Lowest MAE (Priority 1 — 60%)**
- Threshold: MAE ≤ 0.4040 (eliminate baselines immediately)
- Candidates passing: Lasso_Optuna_Standardization (0.4039), Lasso_GridSearch_Standardization (0.4039)
- Eliminated: Lasso_Optuna_Normalization, Lasso_GridSearch_Normalization, both baselines

**Step 2: Verify RMSE Proportional to MAE (Priority 2 — 30%)**
- Both candidates: RMSE = 0.7113 (identical)
- MAE-to-RMSE ratio: 0.4039 / 0.7113 ≈ 0.568 (same for both)
- Status: **No catastrophic divergence detected**. Both candidates pass.

**Step 3: Confirm Acceptable R² (Priority 3 — 10%)**
- Both candidates: R² = 0.9939 (identical, excellent fit)
- Status: Both candidates confirmed acceptable.

**Step 4: Apply Tiebreaker — Lowest Fit Time**
- Lasso_GridSearch_Standardization: 25.50 s ← **LOWER**
- Lasso_Optuna_Standardization: 74.77 s

### **Final Decision**
**Winner: Lasso_GridSearch_Standardization**

**Justification:** GridSearch_Standardization achieves the lowest MAE alongside Optuna_Standardization, both with identical RMSE and R². The tiebreaker rule selects GridSearch_Standardization due to its significantly faster training time (25.50 s vs 74.77 s), making it more efficient for production deployment while maintaining identical predictive performance.